# Hooks, events and effects

You have already written two hooks and been told to skip past them. This is where they stop being scaffolding.

A hook **observes** a run through *events*, and **influences** it by returning *effects*. That is the whole extension model — and almost everything JAZ does to an agent is itself a hook.

## Setup

Install JAZ, provide an API key, and pick a model. Everything below runs in Colab — no local install.

In [1]:
%pip install -q jaz-lang

Note: you may need to restart the kernel to use updated packages.

In [2]:
import os, getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

In [3]:
import jaz
from jaz.llm import LiteLLM

jaz.configure(llm=LiteLLM(model='openai/gpt-5.4-nano'))

## The things you thought were features are hooks

Typed returns, validation, logging, recursion caps, budgets: all one mechanism.

In [4]:
from jaz import invoke
from jaz.hooks import (
    Hook,
    ReturnType,
    ValidateReturn,
    PrintLogger,
    RecursionLimit,
    IterationLimit,
    BudgetPool,
)

for cls in (ReturnType, ValidateReturn, PrintLogger, RecursionLimit, IterationLimit, BudgetPool):
    print(f"{cls.__name__:<16} is a Hook: {issubclass(cls, Hook)}")

ReturnType       is a Hook: True
ValidateReturn   is a Hook: True
PrintLogger      is a Hook: True
RecursionLimit   is a Hook: True
IterationLimit   is a Hook: True
BudgetPool       is a Hook: True


## Hooks are not inputs

This is the distinction worth being precise about.

- An **input** is a keyword argument. It becomes a variable in the agent's REPL and is described in its prompt.
- A **hook** is passed **positionally**. It never enters the REPL and the agent is not told about it. It shapes the run from outside.

`ReturnType` is the one you will reach for first — it constrains what the agent may return:

In [5]:
# Positional, not a keyword argument: a hook is not an input and never becomes a
# variable in the agent's REPL.
total = invoke(ReturnType(int), task="Add 3, 8 and 14.")

print(total, type(total).__name__)

25 int


There are also two *scopes* for a hook, exactly mirroring inputs versus `scope`:

- **Positional** — `invoke(ReturnType(int), ...)` — applies to that one call.
- **As a context manager** — `with BudgetPool(...):` — applies to everything inside, nested invokes included.

## Validating a result

`ValidateReturn` runs your check after the agent returns. If it raises, the message goes back to the agent and it tries again — so the retry is visible:

In [6]:
seen = []


def rejected_once(result):
    """Contrived on purpose: rejects the first answer whatever it is.

    A real validator checks something you care about — a schema, a row count, an
    invariant. This one just makes the retry visible.
    """
    seen.append(result)
    assert len(seen) > 1, "first answer is always rejected; please return it again"


answer = invoke(ReturnType(int), ValidateReturn(rejected_once), task="Return the number 7.")

print(f"attempts: {len(seen)} -> {seen}")
print(f"final answer: {answer}")

attempts: 2 -> [7, 7]
final answer: 7


## Observing: events

A hook handles an event by defining a method named after it: `LLMQueryEnter` becomes `on_llm_query_enter`. There is no registration step.

Return an empty list and you have a pure observer — which is exactly what `PrintUserPrompt` was in tutorial 1:

In [7]:
from collections import Counter


class CountEvents(Hook):
    """Observes only: it returns no effects, so it cannot change the run."""

    def __init__(self):
        self.counts = Counter()

    # A handler is named after its event: LLMQueryEnter -> on_llm_query_enter.
    def on_llm_query_enter(self, event):
        self.counts["llm_query_enter"] += 1
        return []

    def on_repl_exec_exit(self, event):
        self.counts["repl_exec_exit"] += 1
        return []

    def on_invoke_exit(self, event):
        self.counts["invoke_exit"] += 1
        return []


counter = CountEvents()
with counter:
    invoke(ReturnType(int), task="Print 1, then on a later turn print 2, then return 3.")

for name, n in counter.counts.items():
    print(f"{name:<18} {n}")

llm_query_enter    3
repl_exec_exit     3
invoke_exit        1


Those counts *are* the REPL loop, measured: the model was queried, code ran, and that repeated until it returned.

## Influencing: effects

A handler returns a list of **effects** — changes for JAZ to apply. `AddMessages` adds to the conversation; others supply or modify a result, add or drop inputs, or abort the run.

In [8]:
from jaz.hooks.effects import AddMessages


class NudgeAfterFirstTurn(Hook):
    """Injects a message from the second turn onwards."""

    def __init__(self):
        self.fired = 0

    def on_llm_query_enter(self, event):
        if event.iteration < 1:
            return []
        self.fired += 1
        print(f"  hook fired on turn {event.iteration}")
        return [
            AddMessages([
                {"role": "user", "content": "Reminder: return an int as soon as you can."}
            ])
        ]


nudge = NudgeAfterFirstTurn()
with nudge:
    result = invoke(
        ReturnType(int),
        task="Print 1, then on a later turn print 2, then return 3.",
    )

print(f"fired {nudge.fired} time(s); result {result}")

  hook fired on turn 1


fired 1 time(s); result 3


## Limits are hooks too

Which is why they compose the same way. Each of these is set small enough that you can watch it bind — including the recursion cap left over from tutorial 2, which works by simply not giving the agent an `invoke` to call:

In [9]:
from jaz.exceptions import IterationLimitExhaustedError, BudgetPoolExhaustedError

question = "Return True if the name invoke is defined in your REPL, else return False."
print("recursion, no limit    ->", invoke(ReturnType(bool), task=question))
with RecursionLimit(max_depth=1):
    print("recursion, max_depth=1 ->", invoke(ReturnType(bool), task=question))

try:
    with IterationLimit(max_iterations=2):
        invoke(ReturnType(int), task="Print 1, 2, 3, 4, 5 one per turn, then return 5.")
except IterationLimitExhaustedError as exc:
    print("iterations ->", type(exc).__name__)

try:
    with BudgetPool(cost_budget=0.00001):
        invoke(ReturnType(list), task="Write a detailed 10-step database migration plan.")
except BudgetPoolExhaustedError as exc:
    print("budget     ->", str(exc)[:70])

recursion, no limit    -> True


recursion, max_depth=1 -> False


iterations -> IterationLimitExhaustedError


## When a run fails

Those limits raise, which is worth handling deliberately — and there are two different kinds of failure.

**JAZ stopped the run**: a budget, a limit, a return-type violation. Everything JAZ raises descends from `jaz.exceptions.JazException`, so one `except` covers all of it.

**The agent gave up**: asked for something impossible, it raises, and *its* exception reaches you unchanged. That is not a JAZ error, so `except JazException` will not catch it:

In [10]:
from jaz.exceptions import JazException


def run_safely(task):
    """Two kinds of failure, and they arrive differently."""
    try:
        return invoke(ReturnType(int), task=task)
    except JazException as exc:
        # JAZ stopped the run: a budget, a limit, a return-type violation.
        print(f"  JAZ stopped it  -> {type(exc).__name__}")
    except Exception as exc:
        # The agent decided the task was impossible and raised. Its exception is yours.
        print(f"  the agent raised -> {type(exc).__name__}: {str(exc)[:80]}")
    return None


print("a task it can do:")
print("  ", run_safely("Return the number 3."))

print("a task it cannot:")
print("  ", run_safely("Return today's gold price in USD. Do not guess or invent a number."))

a task it can do:


   3
a task it cannot:


  the agent raised -> Exception: Cannot return today's gold price in USD without guessing: this REPL environment 
   None


## What to take away

- **Hooks are the extension model.** Return types, validation, budgets, limits and logging are all the same mechanism.
- **Events in, effects out.** Define `on_<event>` and return a list — empty to observe, non-empty to influence.
- **Positional is local; `with` propagates.**
- **A hook is not an input.** It never reaches the agent's REPL.
- **`JazException` is the base class** for everything JAZ raises; an agent that gives up raises its own exception instead.

---

**Next:** [Configuration](04-configuration.ipynb) — the model, the REPL and the protocol underneath all of this.